**Лабораторная работа №7**

In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, KFold

1. Загрузите объекты из новостного датасета 20 newsgroups, относящиеся к категориям "космос" и "атеизм".

In [2]:
newsgroups = fetch_20newsgroups(subset='all',
                                categories=['alt.atheism', 'sci.space'],
                                shuffle=True, random_state=241)

X_texts = newsgroups.data
y = newsgroups.target

print(f"Всего документов: {len(X_texts)}")
print(f"Распределение классов: {np.bincount(y)}")

Всего документов: 1786
Распределение классов: [799 987]


2. Вычислите TF-IDF-признаки для всех текстов.

In [3]:
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(X_texts)

print("Размер матрицы (документы, признаки):", X_tfidf.shape)

Размер матрицы (документы, признаки): (1786, 28382)


3. Подберите минимальный лучший параметр C из множества [10^−5, 10^−4, ... 10^4, 10^5] для SVM с линейным ядром (kernel=’linear’) при помощи кросс-валидации по 5 блокам. Укажите параметр random_state=241 и для SVM, и для KFold. В качестве меры качества используйте долю верных ответов (accuracy).

In [4]:
C_values = np.power(10.0, np.arange(-5, 6))
param_grid = {'C': C_values}

kf = KFold(n_splits=5, shuffle=True, random_state=241)
svm = SVC(kernel='linear', random_state=241)

gs = GridSearchCV(svm, param_grid, scoring='accuracy', cv=kf, n_jobs=-1)
gs.fit(X_tfidf, y)

print("Лучший параметр C:", gs.best_params_['C'])
print("Лучшая средняя accuracy:", gs.best_score_)

Лучший параметр C: 10.0
Лучшая средняя accuracy: 0.9915997683989797


Анализ: если несколько C дают одинаковое качество, нужно выбрать наименьший.

In [5]:
best_score = gs.best_score_
best_C_candidates = [params['C'] for params, score in zip(gs.cv_results_['params'], gs.cv_results_['mean_test_score'])
                     if score == best_score]

best_C = min(best_C_candidates)
print(f"Максимальное качество: {best_score:.4f}")
print(f"Минимальный лучший C: {best_C}")

Максимальное качество: 0.9916
Минимальный лучший C: 10.0


4. Обучите SVM по всей выборке с лучшим параметром C, найденным
на предыдущем шаге.

In [6]:
final_svm = SVC(kernel='linear', C=best_C, random_state=241)
final_svm.fit(X_tfidf, y)
print("Модель обучена.")

Модель обучена.


5. Найдите 10 слов с наибольшим по модулю весом. Они являются ответом на это задание. Укажите их через запятую, в нижнем регистре, в лексикографическом порядке.

In [8]:
weights = final_svm.coef_.toarray().flatten()

feature_names = vectorizer.get_feature_names_out()

word_weight_pairs = list(zip(feature_names, np.abs(weights)))

word_weight_pairs.sort(key=lambda x: x[1], reverse=True)

top10_words = [word for word, _ in word_weight_pairs[:10]]
print("10 слов с наибольшим по модулю весом:", top10_words)

10 слов с наибольшим по модулю весом: ['space', 'god', 'atheism', 'atheists', 'moon', 'sky', 'religion', 'bible', 'keith', 'nick']


In [10]:
top10_words_lower = [w.lower() for w in top10_words]
top10_words_sorted = sorted(top10_words_lower)

answer_str = ','.join(top10_words_sorted)
print(answer_str)

atheism,atheists,bible,god,keith,moon,nick,religion,sky,space


7. Сохранение ответа в файл

In [11]:
with open('answer.txt', 'w') as f:
    f.write(answer_str)